In [78]:
import pandas as pd
import numpy as np
import json
import os

In [58]:
# чтение csv
data_clients = pd.read_csv('../data/clients/active_clients.csv', encoding='utf-8')
data_month = pd.read_csv('../data/analytics/monthly_turnovers_2015-01-01_2021-12-31.csv', encoding='utf-8')

# подготовка df
# month
df_month = pd.merge(data_month, data_clients, on=['year', 'month'], how='left')
df_month = df_month.sort_values(['instrument_type', 'year', 'month']).copy()

df_month["delta_value_rub_mm"] = (
    df_month
    .groupby("instrument_type")["value_rub_mm"]
    .diff()
    .fillna(0)
)

df_month["delta_num_trades"] = (
    df_month
    .groupby("instrument_type")["num_trades"]
    .diff()
    .fillna(0)
)



In [59]:
# теперь посчитаем коэффициент, задающий влияние прироста числа активных клиентов на средний прирост объемов, количества сделок и комиссий

df_month['value_rub_mm_per_client'] = np.where(
    (df_month['delta_value_rub_mm'] > 0) & (df_month['delta_active_clients'] > 0),
    df_month['delta_value_rub_mm'] / df_month['delta_active_clients'],
    np.nan
)
df_month['num_trades_per_client'] = np.where(
    (df_month['delta_num_trades'] > 0) & (df_month['delta_active_clients'] > 0),
    df_month['delta_num_trades'] / df_month['delta_active_clients'],
    np.nan
    # будет много nan, но ничего страшного: мы никак не сможем использовать данные с delta<0
)
df_month


,instrument_type,year,month,value_rub_mm,value_usd_mm,num_trades,quarter,active_clients,delta_active_clients,delta_value_rub_mm,delta_num_trades,value_rub_mm_per_client,num_trades_per_client
0,futures,2015,1,3.975247e+06,62206.689846,18763192.0,1.0,65298.0,0.0,0.000000e+00,0.0,NaN,NaN
1,futures,2015,2,6.403488e+06,98303.175617,26257197.0,1.0,72626.0,7328.0,2.428241e+06,7494005.0,331.364832,1022.653521
2,futures,2015,3,6.048007e+06,100096.806334,23132965.0,1.0,70713.0,-1913.0,-3.554815e+05,-3124232.0,NaN,NaN
3,futures,2015,4,6.171327e+06,116502.483836,28356177.0,2.0,69966.0,-747.0,1.233210e+05,5223212.0,NaN,NaN
4,futures,2015,5,4.203682e+06,83429.496038,18948506.0,2.0,61889.0,-8077.0,-1.967646e+06,-9407671.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,totals,2021,8,1.176555e+07,159936.128793,26986222.0,3.0,2282708.0,134004.0,-7.657465e+05,-1017592.0,NaN,NaN
248,totals,2021,9,1.271883e+07,174431.902652,29935793.0,3.0,2500010.0,217302.0,9.532845e+05,2949571.0,4.386911,13.573603
249,totals,2021,10,1.317436e+07,184331.537725,32409601.0,4.0,2643278.0,143268.0,4.555256e+05,2473808.0,3.179535,17.266996
250,totals,2021,11,1.757090e+07,241771.318402,44339394.0,4.0,2954939.0,311661.0,4.396546e+06,11929793.0,14.106821,38.278107


In [73]:
# возьмем медианное значение и получим коэффициент
value_rub_mm_growth_per_client_forts = (df_month.loc[df_month['instrument_type'] == 'futures']['value_rub_mm_per_client'].median(skipna=True))
num_trades_growth_per_client_forts = (df_month.loc[df_month['instrument_type'] == 'futures']['num_trades_per_client'].median(skipna=True))

value_rub_mm_growth_per_client_options = (df_month.loc[df_month['instrument_type'] == 'options']['value_rub_mm_per_client'].median(skipna=True))
num_trades_growth_per_client_options = (df_month.loc[df_month['instrument_type'] == 'options']['num_trades_per_client'].median(skipna=True))

print(value_rub_mm_growth_per_client_options)

4.049307496150123


In [75]:
def get_int(year, month, instrument, type):
    data_month = pd.read_csv("../data/analytics/monthly_turnovers_2022-01-01_2026-05-31.csv", encoding="utf-8-sig")
    val = data_month.loc[
    (data_month["year"] == year) & (data_month["month"] == month) & (data_month["instrument_type"] == instrument), type].iloc[0]
    return val

In [83]:
# тот же коэффициент рассчитанный по 2026-04 с сайта московской биржи, но по фактическим клиентам срочного рынка
unique_physical_clients_cur = 158636
unique_physical_clients_prev = (1 - (-9.62/100)) * unique_physical_clients_cur

delta_unique_physical_clients = unique_physical_clients_cur - unique_physical_clients_prev

val_cur_option = get_int(2026, 4, "options", "value_rub_mm")
val_prev_option = get_int(2026, 3, "options", "value_rub_mm")
trades_cur_option = get_int(2026, 4, "options", "num_trades")
trades_prev_option = get_int(2026, 3, "options", "num_trades")

delta_value_rub_mm_option = val_cur_option - val_prev_option
delta_num_trades_option = trades_cur_option - trades_prev_option

val_cur_future = get_int(2026, 4, "futures", "value_rub_mm")
val_prev_future = get_int(2026, 3, "futures", "value_rub_mm")
trades_cur_future = get_int(2026, 4, "futures", "num_trades")
trades_prev_future = get_int(2026, 3, "futures", "num_trades")

delta_value_rub_mm_future = val_cur_future - val_prev_future
delta_num_trades_future = trades_cur_future - trades_prev_future

# собственно, коэффициенты
value_rub_mm_growth_per_client_option_real = delta_value_rub_mm_option / delta_unique_physical_clients

num_trades_growth_per_client_option_real = delta_num_trades_option / delta_unique_physical_clients

value_rub_mm_growth_per_client_future_real = delta_value_rub_mm_future / delta_unique_physical_clients

num_trades_growth_per_client_future_real = delta_num_trades_future / delta_unique_physical_clients


Такие данные дают большую точность прогноза, поскольку мы грубо предполагаем, что влияние от появления новых активных клиентов торгующих именно опционами будет аналогично влиянию от появления новых активных клиентов у брокеров. Поэтому мы также скорректируем наш модельный коэффициент, взяв среднее геометрическое между ним и посчитанным коэффициентом в ячейке выше

Теперь посчитаем, насколько вырастут комиссионные доходы без корректировки через среднее геометрическое и с корректировкой.

In [88]:
# сохраняем в json
clients_params = {
    "description": {
        "value_rub_mm_growth_per_client_forts": "Median monthly turnover growth per client for futures (model, 2015-2021)",
        "num_trades_growth_per_client_forts": "Median monthly trades growth per client for futures (model, 2015-2021)",
        "value_rub_mm_growth_per_client_options": "Median monthly turnover growth per client for options (model, 2015-2021)",
        "num_trades_growth_per_client_options": "Median monthly trades growth per client for options (model, 2015-2021)",
        "value_rub_mm_growth_per_client_future_real": "Monthly turnover growth per client for futures (real, 2026-04)",
        "num_trades_growth_per_client_future_real": "Monthly trades growth per client for futures (real, 2026-04)",
        "value_rub_mm_growth_per_client_option_real": "Monthly turnover growth per client for options (real, 2026-04)",
        "num_trades_growth_per_client_option_real": "Monthly trades growth per client for options (real, 2026-04)"
    },
    "data": {
        "value_rub_mm_growth_per_client_forts": value_rub_mm_growth_per_client_forts,
        "num_trades_growth_per_client_forts": num_trades_growth_per_client_forts,
        "value_rub_mm_growth_per_client_options": value_rub_mm_growth_per_client_options,
        "num_trades_growth_per_client_options": num_trades_growth_per_client_options,
        "value_rub_mm_growth_per_client_future_real": value_rub_mm_growth_per_client_future_real,
        "num_trades_growth_per_client_future_real": num_trades_growth_per_client_future_real,
        "value_rub_mm_growth_per_client_option_real": value_rub_mm_growth_per_client_option_real,
        "num_trades_growth_per_client_option_real": num_trades_growth_per_client_option_real
    }
}


In [89]:
os.makedirs("../data/model_params", exist_ok=True)
file_name = "client_params.json"
with open(f"../data/model_params/{file_name}", "w") as file:
    json.dump(clients_params, file, indent=4) # indent=4 для отступов


Получаем консервативную оценку прироста - на основе данных 2015-2021 годов, и более оптимистичную оценку - на основе данных 2026-04. Посчитаем прирост комиссионных доходов по опционам в обоих сценариях, посчитанных через разные прокси (объемы и количество сделок).

Допустимо делать такое предположение на основе данных о приросте общего количества клиентов с 2015 по 2021 включительно, поскольку на данный момент рынок более развит и покажет большие показатели прироста, чем в среднем за 2015-2021 годы.

Также стоит сказать, что при росте числа активных клиентов опционами будет расти и число клиентов, торгующих фьючерсами. Значит, комиссионные доходы от срочного рынка в целом будут еще выше.

** Возможно, стоит попытаться предсказать и это, но я не знаю, как

Прогноз роста комиссионных будет в отдельном файле